In [8]:
from pathlib import Path
import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

# "v1" = step0.ipynb | "v2" = step0_complex_version.ipynb
SOURCE = "v2"

ENV = "PROD"
START_YEAR, END_YEAR = 1990, 2023   # stessi valori dei notebook di step 0

# File salvati dagli step 0 (nessuna query di settore rilanciata qui)
SECTOR_FILES = {
    "v1": Path("outputs/step0/sector_ids.parquet"),
    "v2": Path(f"outputs/step0_epo/qc_families_v2_{ENV}.csv"),
}


def load_sector_ids(path):
    """Legge parquet o csv e restituisce un DataFrame con la sola colonna 'docdb_family_id'.
    In step0.ipynb la union di SQLAlchemy rinomina la colonna (es. 'anon_1_docdb_family_id'),
    quindi si prende la prima colonna qualunque sia il suo nome."""
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    return df.iloc[:, [0]].set_axis(["docdb_family_id"], axis=1)


sector_ids = load_sector_ids(SECTOR_FILES[SOURCE])

# Output separati per versione, cosi' i due campioni non si sovrascrivono
OUTPUT_DIR = Path(f"outputs/validation_{SOURCE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

print(f"Sorgente: {SOURCE} | file: {SECTOR_FILES[SOURCE]} | famiglie nel settore: {sector_ids['docdb_family_id'].nunique():,}")


Sorgente: v2 | file: outputs/step0_epo/qc_families_v2_PROD.csv | famiglie nel settore: 12,177


In [9]:
N_POS, N_NEG = 50, 50
NEG_MOD, NEG_REMAINDER = 1000, 7   # sottoinsieme deterministico di negativi (in PROD usa 1000 o più)

has_text = or_(TLS202_APPLN_TITLE.appln_title.isnot(None),
               TLS203_APPLN_ABSTR.appln_abstract.isnot(None))


def eligible_families(*filters):
    """Famiglie nel periodo con almeno un'application che ha titolo o abstract (qualsiasi lingua)."""
    return patstat.df(
        db.query(TLS201_APPLN.docdb_family_id.label("docdb_family_id"))
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(TLS201_APPLN.docdb_family_id > 0)
        .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
        .filter(has_text)
        .filter(*filters)
        .filter(TLS201_APPLN.ipr_type == "PI")
        .distinct()
    )


in_sector = TLS201_APPLN.docdb_family_id.in_(sector_ids["docdb_family_id"].dropna().unique().tolist())

# 1. Positivi: famiglie nel perimetro con almeno un testo disponibile
pool_pos = eligible_families(in_sector)
df_pos = pool_pos.sample(n=min(N_POS, len(pool_pos)), random_state=42).copy()
df_pos["query_label"] = 1

# 2. Negativi: famiglie fuori dal perimetro con almeno un testo disponibile
pool_neg = eligible_families(
    ~in_sector,
    TLS201_APPLN.docdb_family_id % NEG_MOD == NEG_REMAINDER
    
)
df_neg = pool_neg.sample(n=min(N_NEG, len(pool_neg)), random_state=42).copy()
df_neg["query_label"] = 0

df_validation = (
    pd.concat([df_pos, df_neg])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
family_ids_list = df_validation["docdb_family_id"].tolist()

# 3. Testi di tutte le application delle famiglie campionate (che hanno titolo o abstract)
text_query = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        func.max(TLS202_APPLN_TITLE.appln_title).label("appln_title"),
        func.max(TLS203_APPLN_ABSTR.appln_abstract).label("appln_abstract"),
    )
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(family_ids_list))
    .filter(has_text)
    .group_by(TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id)
)
df_texts = patstat.df(text_query)

# Una application per famiglia: preferisci quella con abstract, poi con titolo, poi appln_id minore
df_texts["has_abs"] = df_texts["appln_abstract"].notna()
df_texts["has_title"] = df_texts["appln_title"].notna()
df_texts = (
    df_texts.sort_values(["docdb_family_id", "has_abs", "has_title", "appln_id"],
                         ascending=[True, False, False, True])
    .drop_duplicates("docdb_family_id")
    .drop(columns=["has_abs", "has_title"])
)

df_validation = df_validation.merge(df_texts, on="docdb_family_id", how="left")
df_validation["manual_label"] = ""

# CSV completo (con query_label) e CSV in cieco per l'etichettatura
validation_csv_path = OUTPUT_DIR / "validation_sample.csv"
df_validation.to_csv(validation_csv_path, index=False)

print(f"Positivi: {len(df_pos)} | Negativi: {len(df_neg)} | Totale: {len(df_validation)}")
print(f"CSV completo: {validation_csv_path}")
display(df_validation.head(5))

Positivi: 50 | Negativi: 50 | Totale: 100
CSV completo: outputs/validation_v2/validation_sample.csv


,docdb_family_id,query_label,appln_id,appln_title,appln_abstract,manual_label
0,51847007,0,423486124,PROCESS FOR IN SITU UPGRADING OF A HEAVY HYDRO...,Disclosed is a process for in situ upgrading o...,
1,55589007,0,450929721,Providing virtual storage processor (VSP) mobi...,A technique provides virtual storage processor...,
2,9947007,0,7041503,Solvent composition and use thereof,本发明涉及一种在制造光记录介质过程中有用的新的溶剂组合物。本发明提供了一种新的液体组合物，含...,
3,85038492,1,586931638,Performing dynamic programmatic entanglement i...,Performing dynamic programmatic entanglement i...,
4,93938571,1,623446109,"LEARNING DEVICE, LEARNING METHOD, AND PROGRAM",Provided is a learning device for learning a r...,


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df_lab = pd.read_csv(validation_csv_path)   # dopo aver compilato manual_label con 0/1
#y_true = df_lab["manual_label"].astype(int)
# ATTENZIONE: questa lista vale solo per il campione su cui e' stata compilata;
# cambiando SOURCE il campione cambia -> usare manual_label del CSV (riga sopra)
y_true = [0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0]
y_pred = df_lab["query_label"].astype(int)

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1:        {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(confusion_matrix(y_true, y_pred))